# Bab 3: Introduction to Neural Prediction — Forward Propagation

Notebook ini merangkum Bab 3 buku *Grokking Deep Learning* karya Andrew W. Trask. Bab ini mulai membangun neural network pertama dari dasar: satu input/satu weight, lalu bertahap ke banyak input, banyak output, kombinasi keduanya, sampai network bertumpuk (stacked). Semua dibangun memakai Python murni dulu, lalu diterjemahkan ke NumPy.

## 1. Neural Network Paling Sederhana: Satu Input, Satu Weight

Neural network dalam bentuk paling dasarnya adalah satu atau lebih weight yang dikalikan dengan data input untuk menghasilkan prediksi. Contoh pertama di buku ini memprediksi probabilitas menang suatu tim bisbol hanya berdasarkan rata-rata jumlah jari kaki (`# toes`) pemainnya.

In [ ]:
weight = 0.1

def neural_network(input, weight):
    prediction = input * weight
    return prediction

number_of_toes = [8.5, 9.5, 10, 9]
input = number_of_toes[0]
pred = neural_network(input, weight)

print(pred)  # 0.85

Beberapa poin penting dari contoh ini:

- **Input data** adalah angka yang dicatat dari dunia nyata, biasanya sesuatu yang mudah diketahui (suhu hari ini, rata-rata pukulan pemain bisbol, harga saham kemarin).
- **Prediksi** adalah apa yang dikeluarkan network berdasarkan input tersebut.
- Prediksi **tidak selalu benar**. Kalau terlalu tinggi, weight akan disesuaikan supaya memprediksi lebih rendah di lain waktu, dan sebaliknya.
- Network belajar lewat **trial and error**: membuat prediksi, melihat apakah terlalu tinggi atau rendah, lalu menyesuaikan weight.

### Apa yang Sebenarnya Dilakukan Network Ini?

Network ini pada dasarnya mengalikan input dengan weight, atau bisa dibilang men-"scale" input dengan jumlah tertentu. Kalau weight-nya 2, network menggandakan input. Kalau weight-nya 0.01, network membaginya dengan 100.

Weight juga bisa dipikirkan sebagai ukuran sensitivitas antara input dan prediksi, mirip knob volume: makin tinggi weight, makin besar prediksi untuk input yang sama kecilnya. Catatan penting lain: network tidak punya memori apa pun selain input yang diberikan saat itu. Kalau kita beri `number_of_toes[1]` setelah ini, network tidak akan mengingat prediksi sebelumnya.

Neural network juga tidak terbatas memprediksi angka positif saja, weight negatif memungkinkan prediksi dan input bernilai negatif, misalnya memprediksi probabilitas orang memakai jaket berdasarkan suhu di bawah nol.

## 2. Prediksi dengan Banyak Input

Satu datapoint saja (rata-rata jumlah jari kaki) belum tentu prediktor yang baik. Network bisa menerima beberapa input sekaligus supaya membuat prediksi yang lebih matang. Setiap input punya weight-nya sendiri, dan hasil kali tiap pasangan input-weight dijumlahkan menjadi satu prediksi, proses ini disebut **weighted sum** atau **dot product**.

In [ ]:
def w_sum(a, b):
    assert len(a) == len(b)
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

weights = [0.1, 0.2, 0]

def neural_network(input, weights):
    pred = w_sum(input, weights)
    return pred

toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

input = [toes[0], wlrec[0], nfans[0]]
pred = neural_network(input, weights)

print(pred)  # 0.98

### Intuisi di Balik Dot Product

Dot product (weighted sum) pada dasarnya memberi ukuran **kemiripan** antara dua vektor. Beberapa contoh dari buku untuk membangun intuisi ini:

```
a = [0, 1, 0, 1]      w_sum(a, b) = 0
b = [1, 0, 1, 0]      w_sum(b, c) = 1
c = [0, 1, 1, 0]      w_sum(b, d) = 1
d = [.5, 0, .5, 0]    w_sum(c, c) = 2
e = [0, 1, -1, 0]     w_sum(d, d) = .5
                      w_sum(c, e) = 0
```

Weighted sum tertinggi justru muncul ketika dua vektor itu persis identik (`w_sum(c, c)`). Dot product juga bisa dianalogikan sebagai operasi logika:

- **AND**: `w_sum(a, b) = 0` karena tidak ada posisi yang sama-sama bernilai di kedua vektor.
- **AND parsial**: kalau salah satu vektor punya weight 0.5 di posisi yang cocok, skor yang dihasilkan juga ikut 0.5 (bukan 1 penuh). Sifat ini dimanfaatkan saat memodelkan probabilitas.
- **NOT**: weight negatif berperan seperti operator NOT, karena weight positif yang dipasangkan dengan weight negatif akan menurunkan skor.

Dari sini bisa dibangun "bahasa" kasar untuk membaca susunan weight, contohnya `weights = [1, 0, -1]` kira-kira berarti "beri skor tinggi jika input[0] ATAU BUKAN input[2]".

Perlu diingat juga: weight tidak bisa diacak posisinya (urutannya penting), baik nilai weight maupun nilai input sama-sama menentukan dampak akhir terhadap skor, dan weight negatif akan mengurangi skor akhir untuk input yang berpasangan dengannya.

### Versi NumPy

Library NumPy punya fungsi `dot()` bawaan yang menggantikan fungsi `w_sum()` buatan sendiri.

In [ ]:
import numpy as np

weights = np.array([0.1, 0.2, 0])

def neural_network(input, weights):
    pred = input.dot(weights)
    return pred

toes = np.array([8.5, 9.5, 9.9, 9.0])
wlrec = np.array([0.65, 0.8, 0.8, 0.9])
nfans = np.array([1.2, 1.3, 0.5, 1.0])

input = np.array([toes[0], wlrec[0], nfans[0]])
pred = neural_network(input, weights)

print(pred)  # 0.98, sama dengan versi Python murni di atas

## 3. Prediksi dengan Banyak Output

Sebaliknya, network juga bisa membuat beberapa prediksi sekaligus dari satu input saja. Caranya mirip tiga network satu-weight yang berjalan terpisah, masing-masing menerima input yang sama tapi punya weight sendiri-sendiri. Operasi ini disebut **elementwise multiplication**.

In [ ]:
def ele_mul(number, vector):
    output = [0, 0, 0]
    assert len(output) == len(vector)
    for i in range(len(vector)):
        output[i] = number * vector[i]
    return output

weights = [0.3, 0.2, 0.9]

def neural_network(input, weights):
    pred = ele_mul(input, weights)
    return pred

wlrec = [0.65, 0.8, 0.8, 0.9]
input = wlrec[0]
pred = neural_network(input, weights)

print(pred)  # [0.195, 0.13, 0.585] -> prediksi hurt?, win?, sad?

Perhatikan bahwa ketiga prediksi ini (hurt, win, sad) sepenuhnya independen satu sama lain, berbeda dari network dengan banyak input ke satu output, di mana prediksinya memang saling berkaitan. Ini membuat network jenis ini sederhana untuk diimplementasikan, karena masing-masing output pada dasarnya adalah network terpisah yang menerima input yang sama.

## 4. Prediksi dengan Banyak Input dan Output

Keduanya bisa digabung: banyak input dan banyak output sekaligus. Setiap pasangan input-output punya weight-nya sendiri, sehingga kumpulan weight-nya membentuk sebuah **matrix** (daftar vektor). Operasi yang dipakai disebut **vector-matrix multiplication**: tiap baris di matrix weight di-dot-product-kan dengan vektor input.

In [ ]:
           # toes  % win  # fans
weights = [[0.1, 0.1, -0.3],  # hurt?
           [0.1, 0.2, 0.0],   # win?
           [0.0, 1.3, 0.1]]   # sad?

def w_sum(a, b):
    assert len(a) == len(b)
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

def vect_mat_mul(vect, matrix):
    assert len(vect) == len(matrix)
    output = [0, 0, 0]
    for i in range(len(vect)):
        output[i] = w_sum(vect, matrix[i])
    return output

def neural_network(input, weights):
    pred = vect_mat_mul(input, weights)
    return pred

toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

input = [toes[0], wlrec[0], nfans[0]]
pred = neural_network(input, weights)

print(pred)  # [0.555, 0.98, 0.965] -> hurt, win, sad

Cara memandang network ini: tiap node output melakukan weighted sum-nya sendiri secara independen terhadap keseluruhan input. Jadi ini sebenarnya tiga dot product berjalan paralel, hanya saja hasilnya disimpan bersama dalam satu vektor output.

## 5. Prediksi Bertumpuk (Stacking): Hidden Layer

Output dari satu network bisa dijadikan input untuk network berikutnya, menghasilkan dua vector-matrix multiplication berurutan. Lapisan tengah di antara input dan output ini disebut **hidden layer**. Ini belum terlihat manfaatnya secara langsung di contoh sederhana, tapi berguna untuk dataset dengan pola yang terlalu kompleks untuk ditangkap satu matrix weight saja (misalnya klasifikasi gambar).

In [ ]:
import numpy as np

           # toes  % win  # fans
ih_wgt = [[0.1, 0.2, -0.1],   # hid[0]
          [-0.1, 0.1, 0.9],   # hid[1]
          [0.1, 0.4, 0.1]]    # hid[2]

           # hid[0] hid[1] hid[2]
hp_wgt = [[0.3, 1.1, -0.3],   # hurt?
          [0.1, 0.2, 0.0],    # win?
          [0.0, 1.3, 0.1]]    # sad?

weights = [ih_wgt, hp_wgt]

def w_sum(a, b):
    assert len(a) == len(b)
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

def vect_mat_mul(vect, matrix):
    assert len(vect) == len(matrix)
    output = [0, 0, 0]
    for i in range(len(vect)):
        output[i] = w_sum(vect, matrix[i])
    return output

def neural_network(input, weights):
    hid = vect_mat_mul(input, weights[0])
    pred = vect_mat_mul(hid, weights[1])
    return pred

toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

input = [toes[0], wlrec[0], nfans[0]]
pred = neural_network(input, weights)

print(pred)  # [0.214, 0.145, 0.507]

### Versi NumPy

Versi NumPy dari network bertumpuk ini memakai `.dot()` dan transpose (`.T`) agar dimensi matrix weight cocok untuk operasi dot product.

In [ ]:
import numpy as np

           # toes  % win  # fans
ih_wgt = np.array([
    [0.1, 0.2, -0.1],   # hid[0]
    [-0.1, 0.1, 0.9],   # hid[1]
    [0.1, 0.4, 0.1]]).T # hid[2]

           # hid[0] hid[1] hid[2]
hp_wgt = np.array([
    [0.3, 1.1, -0.3],   # hurt?
    [0.1, 0.2, 0.0],    # win?
    [0.0, 1.3, 0.1]]).T # sad?

weights = [ih_wgt, hp_wgt]

def neural_network(input, weights):
    hid = input.dot(weights[0])
    pred = hid.dot(weights[1])
    return pred

toes = np.array([8.5, 9.5, 9.9, 9.0])
wlrec = np.array([0.65, 0.8, 0.8, 0.9])
nfans = np.array([1.2, 1.3, 0.5, 1.0])

input = np.array([toes[0], wlrec[0], nfans[0]])
pred = neural_network(input, weights)

print(pred)

## 6. Primer Singkat NumPy

Beberapa aturan dasar yang penting dipahami saat membaca kode NumPy:

- Membuat vektor dan matrix prosesnya sama persis: matrix dengan satu baris saja adalah vektor. Urutan dimensinya selalu (baris, kolom).
- Operasi elementwise (`+`, `-`, `*`, `/`) mengharuskan kedua variabel punya jumlah kolom yang sama, **atau** salah satunya hanya punya satu kolom (sehingga di-broadcast).
- `a * 0.1` (vektor dikali skalar) akan mengalikan skalar itu ke semua elemen vektor.
- `a * b` (dua vektor dengan jumlah kolom sama) menghasilkan perkalian elementwise berdasarkan posisi.
- `a * c` (vektor dikali matrix dengan jumlah kolom yang sama) akan mengalikan vektor itu ke setiap baris matrix.
- Jika dimensi tidak cocok sama sekali, NumPy melempar `ValueError: operands could not be broadcast together`.

In [ ]:
import numpy as np

a = np.array([0, 1, 2, 3])
b = np.array([4, 5, 6, 7])
c = np.array([[0, 1, 2, 3],
              [4, 5, 6, 7]])
d = np.zeros((2, 4))
e = np.random.rand(2, 5)

print(a)
print(b)
print(c)
print(d)
print(e)

print(a * 0.1)   # vector-scalar multiplication
print(c * 0.2)   # scalar-matrix multiplication
print(a * b)     # elementwise multiplication
print(a * b * 0.2)
print(a * c)     # a dikalikan ke setiap baris c

### Aturan Emas Dot Product

Satu aturan emas saat memakai `.dot()`: kalau kita tuliskan bentuk `(rows, cols)` dari dua variabel yang di-dot-kan bersisian, angka yang bersebelahan harus sama. Misalnya men-dot-kan `(1,4)` dengan `(4,3)` akan menghasilkan `(1,3)`. Secara umum: `(a,b).dot(b,c) = (a,c)`.

In [ ]:
import numpy as np

a = np.zeros((1, 4))
b = np.zeros((4, 3))
c = a.dot(b)
print(c.shape)  # (1, 3)

a2 = np.zeros((2, 4))
b2 = np.zeros((4, 3))
c2 = a2.dot(b2)
print(c2.shape)  # (2, 3)

e = np.zeros((2, 1))
f = np.zeros((1, 3))
g = e.dot(f)
print(g.shape)  # (2, 3)

# .T membalik baris dan kolom suatu matrix
h = np.zeros((5, 4)).T  # jadi (4, 5)
i_ = np.zeros((5, 6))
j = h.dot(i_)
print(j.shape)  # (4, 6)

## Poin-poin Penting Bab 3

- Neural network, pada intinya, membuat prediksi lewat weighted sum (dot product) yang berulang antara input dan weight.
- Proses ini disebut **forward propagation**: mengalirkan aktivasi (semua angka yang bukan weight) ke depan melewati network.
- Dot product memberi ukuran kemiripan antara dua vektor, dan bisa dianalogikan dengan operasi logika AND/NOT.
- Network bisa menerima banyak input (weighted sum), menghasilkan banyak output (elementwise multiplication), atau keduanya (vector-matrix multiplication, memakai matrix weight).
- Network bisa ditumpuk lewat hidden layer, menghasilkan dua vector-matrix multiplication berurutan, berguna untuk pola data yang lebih kompleks.
- NumPy menyederhanakan semua operasi ini lewat `np.array`, `.dot()`, dan broadcasting, tapi tetap perlu memperhatikan bentuk (shape) tiap vektor/matrix agar operasinya valid.
- Bab berikutnya akan membahas bagaimana cara mengatur nilai weight supaya network membuat prediksi yang akurat (gradient descent).

Referensi: Trask, A. W. (2019). *Grokking Deep Learning*, Bab 3 "Introduction to Neural Prediction: Forward Propagation". Manning Publications.